## 1. 기본 패키지 불러오기
ML feature process에 필요한 패키지를 불러옵니다. 기존 처리 로직과 경로는 바꾸지 않았습니다.

In [1]:
# 이 셀은 ML 산출물 생성에 필요한 패키지만 불러옵니다.
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import polars as pl

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


polars: 1.35.2
pandas: 2.2.2
numpy: 2.0.2


## 2. 팀 공통 Google Drive 경로 설정
02번 preprocessing에서 생성한 `clean_base.parquet`을 읽고, ML용 산출물을 저장할 위치를 지정합니다.

In [2]:
from pathlib import Path

# ============================================================
# Google Drive 경로 설정
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
#
# Drive 폴더 구조:
# Graph_AML/
# └── data/
#     ├── raw/
#     ├── processed/
#     │   ├── clean_base/
#     │   └── ml_features/
#     │   └── gnn_inputs/
#
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# 모든 산출물은 Google Drive에 저장합니다.
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"

CLEAN_BASE_DIR = PROCESSED_DIR / "clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = PROCESSED_DIR / "gnn_inputs"
VALIDATION_DIR = ML_DRIVE_DIR / "validation"


# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
CLEAN_BASE_DIR = CLEAN_BASE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    RAW_DIR,
    PROCESSED_DIR,
    CLEAN_BASE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
    VALIDATION_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("CLEAN_BASE_DIR :", CLEAN_BASE_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)
CONFIG = {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,

    # 02번 산출물 위치 후보. 새 Drive 구조를 우선 사용합니다.
    "CLEAN_BASE_CANDIDATES": [
        CLEAN_BASE_DIR / "clean_base.parquet",
        PROCESSED_DIR / "clean_base.parquet",
    ],

    # 큰 parquet 산출물은 Drive에 저장
    "DRIVE_OUTPUT_DIR": ML_DRIVE_DIR,


    # 기존 코드 호환용: 03번의 주 산출물 위치는 Drive
    "OUTPUT_DIR": ML_DRIVE_DIR,
    "VALIDATION_DIR": VALIDATION_DIR,

    "RANDOM_SEED": 42,

    "SAMPLE_MODE": True,

    "TRAIN_RATIO": 0.60,
    "VAL_RATIO": 0.20,
    "TEST_RATIO": 0.20,

    "WINDOWS": {
        "w1h": "1h",
        "w6h": "6h",
        "w1d": "1d",
        "w3d": "3d",
        "w7d": "7d",
    },

    # Small 데이터에서는 30d를 기본 feature로 만들지 않음
    "MAKE_OPTIONAL_30D": False,

    "COLUMN_CANDIDATES": {
        "timestamp": [
            "timestamp", "Timestamp", "time", "Time", "datetime", "DateTime"
        ],
        "label": [
            "is_laundering", "Is_Laundering", "label", "target", "y"
        ],
        "amount": [
            "amount", "Amount", "amount_paid", "Amount Paid", "paid_amount"
        ],

        # 02번 표준 컬럼명을 우선한다.
        "sender_account": [
            "sender_account_id", "sender_account", "from_account", "Account", "account",
            "orig_account", "source_account", "From Account"
        ],
        "receiver_account": [
            "receiver_account_id", "receiver_account", "to_account", "Account.1",
            "beneficiary_account", "dest_account", "destination_account", "To Account"
        ],
        "from_bank": [
            "sender_bank_id", "from_bank", "From Bank", "sender_bank", "source_bank"
        ],
        "to_bank": [
            "receiver_bank_id", "to_bank", "To Bank", "receiver_bank", "destination_bank"
        ],

        "payment_currency": [
            "payment_currency", "Payment Currency", "currency", "Currency"
        ],
        "receiving_currency": [
            "receiving_currency", "Receiving Currency", "received_currency"
        ],

        "payment_format": [
            "payment_format", "Payment Format", "payment_type", "Payment Type", "format"
        ],
        "tx_id": [
            "tx_id", "transaction_id", "TransactionID", "transaction_id_raw"
        ],
    },

    # 모델 입력 feature 점검용.
    # exact forbidden은 컬럼명 전체 일치만 금지하고
    # substring forbidden은 라벨/그래프 누수성 토큰만 검사한다.
    "EXACT_FORBIDDEN_MODEL_COLUMNS": {
        "label", "is_laundering", "tx_id", "transaction_id",
        "split", "timestamp", "datetime",
    },
    "SUBSTRING_FORBIDDEN_MODEL_TOKENS": {
        "laundering", "attempt_id", "pattern",
        "pagerank", "centrality", "degree",
    },
}

CONFIG["DRIVE_OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", CONFIG["BASE_DIR"])
print("DATA_DIR:", CONFIG["DATA_DIR"])
print("DRIVE_OUT:", CONFIG["DRIVE_OUTPUT_DIR"])
print("clean_base candidates:")
for p in CONFIG["CLEAN_BASE_CANDIDATES"]:
    print(" -", p, "exists=", Path(p).exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_BASE_DIR   : /content/drive/MyDrive/Graph_AML
DRIVE_DATA_DIR   : /content/drive/MyDrive/Graph_AML/data
RAW_DIR          : /content/drive/MyDrive/Graph_AML/data/raw
CLEAN_BASE_DIR : /content/drive/MyDrive/Graph_AML/data/processed/clean_base
ML_DRIVE_DIR     : /content/drive/MyDrive/Graph_AML/data/processed/ml_features
GNN_DRIVE_DIR    : /content/drive/MyDrive/Graph_AML/data/processed/gnn_inputs
BASE_DIR: /content/drive/MyDrive/Graph_AML
DATA_DIR: /content/drive/MyDrive/Graph_AML/data
DRIVE_OUT: /content/drive/MyDrive/Graph_AML/data/processed/ml_features
clean_base candidates:
 - /content/drive/MyDrive/Graph_AML/data/processed/clean_base/clean_base.parquet exists= True
 - /content/drive/MyDrive/Graph_AML/data/processed/clean_base.parquet exists= False


## 3. 보조 함수 준비
split, feature catalog, leakage check 등에 반복해서 쓰는 함수입니다. 산출물의 신뢰도를 높이기 위한 검증 보조 코드입니다.

In [3]:
# ============================================================
# 보조 함수
# ============================================================

MEMORY_RECORDS = []

# 후보 경로 중 실제 존재하는 파일을 선택합니다.
def resolve_existing_path(candidates, name="input"):
    for path in candidates:
        if Path(path).exists():
            print("{} path: {}".format(name, path))
            return Path(path)

    msg = "{} 파일을 찾지 못했습니다. 확인한 후보 경로:\n{}".format(
        name,
        "\n".join([str(p) for p in candidates])
    )
    raise FileNotFoundError(msg)


def get_memory_mb():
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024 / 1024
    except Exception:
        return np.nan


# 주요 단계별 메모리 사용량을 리포트용으로 기록합니다.
def record_memory(step, df=None):
    row_count = None
    col_count = None

    if df is not None:
        row_count = df.height
        col_count = len(df.columns)

    MEMORY_RECORDS.append({
        "step": step,
        "row_count": row_count,
        "col_count": col_count,
        "memory_mb": get_memory_mb(),
        "checked_at": pd.Timestamp.now().isoformat(),
    })

    print("[MEM]", step, "| rows=", row_count, "| cols=", col_count, "| mb=", round(get_memory_mb(), 2))


# 누적된 메모리 기록을 CSV로 저장합니다.
def save_memory_profile():
    path = CONFIG["VALIDATION_DIR"] / "memory_profile.csv"
    pd.DataFrame(MEMORY_RECORDS).to_csv(path, index=False, encoding="utf-8-sig")
    print("saved:", path)


def resolve_column(df, logical_name, required=True):
    candidates = CONFIG["COLUMN_CANDIDATES"].get(logical_name, [])
    existing = set(df.columns)

    for c in candidates:
        if c in existing:
            return c

    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    if required:
        raise ValueError(
            "필수 컬럼을 찾지 못했습니다. logical_name={}, candidates={}, columns={}".format(
                logical_name, candidates, df.columns
            )
        )

    return None


def safe_log1p_expr(col):
    return (
        pl.when(pl.col(col).is_null() | (pl.col(col) < 0))
        .then(None)
        .otherwise(pl.col(col).log1p())
    )


def parse_window_to_ns(window_str):
    unit = window_str[-1]
    value = int(window_str[:-1])

    if unit == "h":
        seconds = value * 3600
    elif unit == "d":
        seconds = value * 24 * 3600
    elif unit == "m":
        seconds = value * 60
    else:
        raise ValueError("지원하지 않는 window입니다: {}".format(window_str))

    return int(seconds * 1_000_000_000)


def series_datetime_to_ns(series):
    return series.to_numpy().astype("datetime64[ns]").astype("int64")


def safe_schema_get(schema, col, default="unknown"):
    return str(schema[col]) if col in schema else default


def print_shape(name, df):
    print("{}: rows={:,}, cols={:,}".format(name, df.height, len(df.columns)))

## 4. clean_base 로드
02번에서 만든 공통 기준 데이터를 읽습니다. 이 단계부터는 원본 CSV가 아니라 clean base를 기준으로 ML 파일을 만듭니다.

In [4]:
# ============================================================
# clean_base 로드 및 기본 컬럼 정리
# ============================================================

# 02번 산출물 후보 중 실제 존재하는 clean_base를 읽습니다.
clean_path = resolve_existing_path(CONFIG["CLEAN_BASE_CANDIDATES"], name="clean_base")

df_raw = pl.read_parquet(clean_path)
record_memory("load_clean_base", df_raw)
print_shape("df_raw", df_raw)

COL = {
    "timestamp": resolve_column(df_raw, "timestamp", required=True),
    "label": resolve_column(df_raw, "label", required=True),
    "amount": resolve_column(df_raw, "amount", required=True),
    "sender_account": resolve_column(df_raw, "sender_account", required=True),
    "receiver_account": resolve_column(df_raw, "receiver_account", required=True),
    "from_bank": resolve_column(df_raw, "from_bank", required=False),
    "to_bank": resolve_column(df_raw, "to_bank", required=False),
    "payment_currency": resolve_column(df_raw, "payment_currency", required=False),
    "receiving_currency": resolve_column(df_raw, "receiving_currency", required=False),
    "payment_format": resolve_column(df_raw, "payment_format", required=False),
    "tx_id": resolve_column(df_raw, "tx_id", required=False),
}

print("매칭된 컬럼")
for k, v in COL.items():
    print("-", k, ":", v)

# clone 없이 바로 처리
df = df_raw

# 원본 clean_base에 없어서 unknown 상수로 만든 범주형 컬럼을 추적합니다.
# 이런 컬럼은 실제 원본 신호가 아니므로 인코딩 대상에서 제외합니다.
CREATED_UNKNOWN_CATEGORICAL_COLS = []

# tx_id 없으면 row 순서 기준 생성
if COL["tx_id"] is None:
    df = df.with_row_index("tx_id")
    COL["tx_id"] = "tx_id"

# 은행 / 통화 / 결제방식 컬럼 없으면 unknown
for key in ["from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"]:
    if COL[key] is None:
        # 원본 clean_base에 없는 컬럼은 unknown 상수로 만들어 downstream 오류를 막되,
        # 실제 ML 인코딩 대상에서는 제외할 수 있도록 별도 기록합니다.
        df = df.with_columns(pl.lit("unknown").alias(key))
        COL[key] = key
        CREATED_UNKNOWN_CATEGORICAL_COLS.append(key)

# split 라벨을 원본 row에 붙여 이후 산출물에서 공통으로 사용합니다.
df = df.with_columns([
    pl.col(COL["tx_id"]).cast(pl.Utf8).alias("tx_id"),
    pl.col(COL["timestamp"]).cast(pl.Datetime).alias("timestamp"),
    pl.col(COL["label"]).cast(pl.Int64).alias("label"),
    pl.col(COL["amount"]).cast(pl.Float64).alias("amount"),

    pl.col(COL["sender_account"]).cast(pl.Utf8).alias("sender_account"),
    pl.col(COL["receiver_account"]).cast(pl.Utf8).alias("receiver_account"),

    pl.col(COL["from_bank"]).cast(pl.Utf8).fill_null("unknown").alias("from_bank"),
    pl.col(COL["to_bank"]).cast(pl.Utf8).fill_null("unknown").alias("to_bank"),
    pl.col(COL["payment_currency"]).cast(pl.Utf8).fill_null("unknown").alias("payment_currency"),
    pl.col(COL["receiving_currency"]).cast(pl.Utf8).fill_null("unknown").alias("receiving_currency"),
    pl.col(COL["payment_format"]).cast(pl.Utf8).fill_null("unknown").alias("payment_format"),
])

# 02번에서도 label을 한 번 더 확인해 잘못된 입력을 막습니다.
bad_label_df = df.filter(
    pl.col("label").is_null() |
    (~pl.col("label").is_in([0, 1]))
)

invalid_label_count = bad_label_df.height
null_timestamp_count = df.filter(pl.col("timestamp").is_null()).height
null_amount_count = df.filter(pl.col("amount").is_null()).height

print("invalid_label_count:", invalid_label_count)
print("null_timestamp_count:", null_timestamp_count)
print("null_amount_count:", null_amount_count)

if invalid_label_count > 0:
    bad_label_path = CONFIG["VALIDATION_DIR"] / "bad_label_examples.csv"
    bad_label_df.head(50).write_csv(bad_label_path)
    raise ValueError(
        "label이 null 또는 0/1 외 값입니다. rows={}, examples={}".format(
            invalid_label_count,
            bad_label_path,
        )
    )

if null_timestamp_count > 0:
    raise ValueError("timestamp null row가 있습니다.")
if null_amount_count > 0:
    raise ValueError("amount null row가 있습니다.")

df = df.sort(["timestamp", "tx_id"])


record_memory("canonical_schema_ready", df)

df.select([
    "tx_id", "timestamp", "label", "amount",
    "sender_account", "receiver_account",
    "from_bank", "to_bank", "payment_currency", "receiving_currency", "payment_format"
]).head(5)

clean_base path: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/clean_base.parquet
[MEM] load_clean_base | rows= 5078345 | cols= 23 | mb= 1673.83
df_raw: rows=5,078,345, cols=23
매칭된 컬럼
- timestamp : timestamp
- label : is_laundering
- amount : amount
- sender_account : sender_account_id
- receiver_account : receiver_account_id
- from_bank : sender_bank_id
- to_bank : receiver_bank_id
- payment_currency : Payment Currency
- receiving_currency : Receiving Currency
- payment_format : Payment Format
- tx_id : tx_id
invalid_label_count: 0
null_timestamp_count: 0
null_amount_count: 0
[MEM] canonical_schema_ready | rows= 5078345 | cols= 31 | mb= 5728.21


tx_id,timestamp,label,amount,sender_account,receiver_account,from_bank,to_bank,payment_currency,receiving_currency,payment_format
str,datetime[μs],i64,f64,str,str,str,str,str,str,str
"""0""",2022-09-01 00:00:00,0,14675.57,"""03209|8000F4670""","""03209|8000F4670""","""03209""","""03209""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1""",2022-09-01 00:00:00,0,897.37,"""01420|8005DFEB0""","""01420|8005DFEB0""","""01420""","""01420""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""10""",2022-09-01 00:00:00,0,12971.84,"""03284|800146EE0""","""03284|800146EE0""","""03284""","""03284""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""100""",2022-09-01 00:00:00,0,15.69,"""01601|80056E5C0""","""01601|80056E5C0""","""01601""","""01601""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1000""",2022-09-01 00:00:00,0,924649.12,"""019474|803ACC410""","""019474|803ACC410""","""019474""","""019474""","""US Dollar""","""US Dollar""","""Reinvestment"""


## 5. split 확정

아래 우선순위로 split을 확정합니다.

1. `clean_base.parquet`에 `split` 컬럼이 있으면 그대로 사용
2. `split_assignment.csv`가 있으면 tx_id 기준으로 복원하여 사용
3. 기존 단일 산출물 `ml_exp00.parquet`에 `split` 컬럼이 있으면 tx_id 기준으로 복원하여 사용
4. 구버전 train/val/test 분리 parquet만 있으면 tx_id 기준으로 복원하여 사용
5. 위 항목이 모두 없으면 시간순 60/20/20 split을 한 번만 생성하고 `split_assignment.csv`로 저장

**part별·실험별로 split을 다시 생성하는 로직은 금지합니다.**  
ml_exp00, ml_exp00_no_time은 모두 동일한 split을 사용합니다.


In [5]:
# ============================================================
# split 확정 (기존 구조 유지 + 재사용 우선)
#
# [1순위] clean_base에 split 컬럼이 있으면 그대로 사용
# [2순위] split_assignment.csv가 있으면 tx_id 기준 복원
# [3순위] 단일 산출물 ml_exp00.parquet이 있으면 tx_id 기준 복원
# [4순위] 구버전 ml_exp00_train/val/test.parquet이 있으면 tx_id 기준 복원
# [5순위] 모두 없으면 시간순 60/20/20 split을 한 번만 생성
#         -> split_assignment.csv 저장 (이후 재실행 시 2순위로 재사용)
#
# 주의:
#   - part별 / 실험별 split 재생성 금지
#   - ml_exp00 / ml_exp00_no_time 모두 동일 split 사용
# ============================================================

import polars as pl
import numpy as np

# ── 시간순 split 생성 함수 (5순위에서만 호출) ──────────────────────
def _make_time_split_once(df):
    """시간순 60/20/20 split. 같은 timestamp는 같은 split에 배치."""
    if 'timestamp' not in df.columns:
        raise ValueError('[SPLIT] timestamp 컬럼이 없어 시간순 split 생성 불가')
    n = df.height
    train_target = n * CONFIG['TRAIN_RATIO']
    val_target   = n * (CONFIG['TRAIN_RATIO'] + CONFIG['VAL_RATIO'])
    ts_counts = (
        df.group_by('timestamp').agg(pl.len().alias('n_rows'))
          .sort('timestamp')
          .with_columns(pl.col('n_rows').cum_sum().alias('cum_rows'))
    )
    cum = ts_counts['cum_rows'].to_numpy()
    timestamps = ts_counts['timestamp'].to_list()
    if len(timestamps) < 3:
        raise ValueError('[SPLIT] timestamp 종류가 3개 미만 — split 생성 불가')

    def _boundary(target):
        idx = int(np.searchsorted(cum, target, side='left'))
        idx = max(1, min(idx, len(timestamps) - 1))
        if idx > 0 and abs(cum[idx-1]-target) <= abs(cum[idx]-target):
            return idx
        return idx + 1 if idx < len(timestamps)-1 else idx

    tr_end = max(1, min(_boundary(train_target), len(timestamps)-2))
    va_end = max(tr_end+1, min(_boundary(val_target), len(timestamps)-1))
    print(f'[SPLIT NEW] train_last_ts={timestamps[tr_end-1]}')
    print(f'[SPLIT NEW] val_last_ts  ={timestamps[va_end-1]}')
    return df.with_columns(
        pl.when(pl.col('timestamp') <= timestamps[tr_end-1]).then(pl.lit('train'))
          .when(pl.col('timestamp') <= timestamps[va_end-1]).then(pl.lit('val'))
          .otherwise(pl.lit('test'))
          .alias('split')
    )

# ── split 유효성 체크 함수 ──────────────────────────────────────────
def _is_valid_split(df):
    if 'split' not in df.columns:
        return False
    if df.filter(pl.col('split').is_null()).height > 0:
        return False
    vals = set(df['split'].unique().to_list())
    return vals == {'train', 'val', 'test'}

def _restore_split_by_tx_id(df, restore_df, source_name):
    """restore_df(tx_id, split)를 df에 merge하고 검증한다."""
    need_cols = {'tx_id', 'split'}
    if not need_cols <= set(restore_df.columns):
        raise ValueError(f'[SPLIT ERROR] {source_name}에 tx_id 또는 split 컬럼이 없습니다.')
    restore_df = restore_df.select([
        pl.col('tx_id').cast(pl.Utf8),
        pl.col('split').cast(pl.Utf8),
    ])
    bad_vals = set(restore_df['split'].unique().to_list()) - {'train', 'val', 'test'}
    if bad_vals:
        raise ValueError(f'[SPLIT ERROR] {source_name} split 값 이상: {sorted(bad_vals)}')
    if 'split' in df.columns:
        df = df.drop('split')
    df = df.with_columns(pl.col('tx_id').cast(pl.Utf8))
    df = df.join(restore_df, on='tx_id', how='left')
    n_null = df.filter(pl.col('split').is_null()).height
    if n_null > 0:
        raise ValueError(
            f'[SPLIT ERROR] {source_name} join 후 split null {n_null:,}건 — tx_id 불일치 가능성\n'
            '해결 방법:\n'
            '1. 현재 clean_base가 기존 산출물을 만들 때 사용한 clean_base와 같은지 확인하세요.\n'
            '2. 기존 ml_exp00*.parquet가 오래된 산출물이라면 백업/삭제 후 재실행하세요.\n'
            '3. 새 split을 생성하려면 기존 ml_exp00*.parquet와 split_assignment.csv를 모두 제거한 뒤 재실행하세요.'
        )
    return df

_SPLIT_ASSIGN_PATH = CONFIG['DRIVE_OUTPUT_DIR'] / 'split_assignment.csv'
_SPLIT_SOURCE = None

# ── [1순위] clean_base에 split 컬럼이 있는 경우 ────────────────────
if _is_valid_split(df):
    _SPLIT_SOURCE = 'clean_base_column'
    print('[SPLIT 1] clean_base에 split 컬럼이 있습니다. 그대로 사용합니다.')

# ── [2순위] split_assignment.csv가 있는 경우 ───────────────────────
elif _SPLIT_ASSIGN_PATH.exists():
    print('[SPLIT 2] split_assignment.csv 발견 → tx_id 기준으로 split 복원합니다.')
    _assign_df = pl.read_csv(_SPLIT_ASSIGN_PATH)
    df = _restore_split_by_tx_id(df, _assign_df, 'split_assignment.csv')
    _SPLIT_SOURCE = 'split_assignment_csv'

# ── [3순위] 단일 산출물 ml_exp00.parquet이 있는 경우 ───────────────
elif (CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet').exists():
    print('[SPLIT 3] 단일 ml_exp00.parquet 발견 → tx_id 기준으로 split 복원합니다.')
    _pq = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
    _restore_df = pl.read_parquet(_pq, columns=['tx_id', 'split'])
    df = _restore_split_by_tx_id(df, _restore_df, 'ml_exp00.parquet')
    _SPLIT_SOURCE = 'existing_ml_exp00_full_parquet'

# ── [4순위] 구버전 train/val/test 분리 parquet이 있는 경우 ─────────
elif all((CONFIG['DRIVE_OUTPUT_DIR'] / f'ml_exp00_{s}.parquet').exists()
         for s in ['train', 'val', 'test']):
    print('[SPLIT 4] 구버전 ml_exp00 train/val/test parquet 발견 → tx_id 기준으로 split 복원합니다.')
    _split_frames = []
    for _sn in ['train', 'val', 'test']:
        _pq = CONFIG['DRIVE_OUTPUT_DIR'] / f'ml_exp00_{_sn}.parquet'
        _sub = pl.read_parquet(_pq, columns=['tx_id'])
        _sub = _sub.with_columns([
            pl.col('tx_id').cast(pl.Utf8),
            pl.lit(_sn).alias('split'),
        ])
        _split_frames.append(_sub)
    _restore_df = pl.concat(_split_frames)
    df = _restore_split_by_tx_id(df, _restore_df, 'ml_exp00_train/val/test.parquet')
    _SPLIT_SOURCE = 'existing_ml_exp00_split_parquets'

# ── [5순위] 기존 split 산출물이 없으면 시간순 split 한 번 생성 ─────
else:
    print('[SPLIT 5] 기존 split 없음 → 03번에서 시간순 split을 한 번 생성합니다.')
    df = _make_time_split_once(df)
    _SPLIT_SOURCE = 'new_time_split_once'

# ── 최종 검증 + 출력 ─────────────────────────────────────────────────
if not _is_valid_split(df):
    raise ValueError('[SPLIT ERROR] split 확정 후에도 train/val/test 유효성 검증 실패')

split_summary = (
    df.group_by('split')
      .agg([
          pl.len().alias('n_rows'),
          pl.col('timestamp').min().alias('timestamp_min'),
          pl.col('timestamp').max().alias('timestamp_max'),
          pl.col('label').sum().alias('n_positive') if 'label' in df.columns else pl.lit(None).alias('n_positive'),
      ])
      .with_columns((pl.col('n_positive') / pl.col('n_rows')).alias('positive_ratio'))
      .sort(pl.col('split').replace({'train': 0, 'val': 1, 'test': 2}))
)

print(f'[SPLIT OK] source={_SPLIT_SOURCE}')
display(split_summary.to_pandas())


[SPLIT 2] split_assignment.csv 발견 → tx_id 기준으로 split 복원합니다.
[SPLIT OK] source=split_assignment_csv


,split,n_rows,timestamp_min,timestamp_max,n_positive,positive_ratio
0,train,3046861,2022-09-01 00:00:00,2022-09-06 13:35:00,2297,0.000754
1,val,1015920,2022-09-06 13:36:00,2022-09-08 16:12:00,1083,0.001066
2,test,1015564,2022-09-08 16:13:00,2022-09-18 16:18:00,1797,0.001769


In [6]:
# ============================================================
# split 심화 검증
# - tx_id 단위 split 유일성 확인
# - 시간 단조성 확인 (train < val < test)
# ============================================================

# tx_id 단위 유일성
_tx_multi = (
    df.group_by('tx_id').agg(pl.col('split').n_unique().alias('n_split'))
      .filter(pl.col('n_split') > 1).height
)
if _tx_multi > 0:
    raise ValueError(f'[SPLIT ERROR] tx_id가 여러 split에 걸쳐 있음: {_tx_multi}건')

# 시간 단조성: train max ≤ val min, val max ≤ test min
_ts = {
    row['split']: row
    for row in df.group_by('split').agg(
        pl.col('timestamp').min().alias('ts_min'),
        pl.col('timestamp').max().alias('ts_max'),
    ).to_dicts()
}
if all(s in _ts for s in ['train', 'val', 'test']):
    assert _ts['train']['ts_max'] <= _ts['val']['ts_min'], (
        f'[SPLIT ERROR] train max > val min: '
        f'{_ts["train"]["ts_max"]} > {_ts["val"]["ts_min"]}'
    )
    assert _ts['val']['ts_max'] <= _ts['test']['ts_min'], (
        f'[SPLIT ERROR] val max > test min: '
        f'{_ts["val"]["ts_max"]} > {_ts["test"]["ts_min"]}'
    )
    print('[SPLIT OK] 시간 단조성 확인: train ≤ val ≤ test')

# split_assignment.csv 저장 (1순위·3순위·4순위 모두 동일하게 보존)
# 이미 저장된 경우에도 현재 df의 split과 일치하는지 보장하기 위해 갱신
if _SPLIT_SOURCE != 'split_assignment_csv':  # 2순위는 이미 CSV에서 읽었으므로 스킵
    df.select(['tx_id', 'split']).write_csv(_SPLIT_ASSIGN_PATH)
    print(f'[SPLIT] split_assignment.csv 갱신: {_SPLIT_ASSIGN_PATH}')

print(f'[SPLIT DEEP OK] source={_SPLIT_SOURCE}')
print('[SPLIT DEEP OK] ml_exp00 / ml_exp00_no_time 모두 이 split을 공유합니다.')


[SPLIT OK] 시간 단조성 확인: train ≤ val ≤ test
[SPLIT DEEP OK] source=split_assignment_csv
[SPLIT DEEP OK] ml_exp00 / ml_exp00_no_time 모두 이 split을 공유합니다.


## 6. split summary & baseline feature 구성

split별 row 수, 라벨 분포, timestamp 범위를 기록합니다.  
이어서 현재 거래에서 바로 추출할 수 있는 baseline feature를 구성합니다.

In [7]:
# ============================================================
# split_summary.csv
# ============================================================

# split별 row 수, 라벨 분포, timestamp 범위를 요약합니다.
def make_split_summary(df):
    summary = (
        df.group_by("split")
        .agg([
            pl.len().alias("row_count"),
            (pl.col("label") == 0).sum().alias("label_0_count"),
            (pl.col("label") == 1).sum().alias("label_1_count"),
            pl.col("label").mean().alias("label_1_ratio"),
            pl.col("timestamp").min().alias("timestamp_min"),
            pl.col("timestamp").max().alias("timestamp_max"),
        ])
    )

    order = pl.DataFrame({
        "split": ["train", "val", "test"],
        "split_order": [0, 1, 2],
    })

    return (
        summary.join(order, on="split", how="left")
        .sort("split_order")
        .drop("split_order")
    )


# split 결과는 별도 CSV로 저장합니다.
split_summary = make_split_summary(df)
split_summary_path = CONFIG["DRIVE_OUTPUT_DIR"] / "split_summary.csv"
split_summary.write_csv(split_summary_path)

print("saved:", split_summary_path)
split_summary

saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/split_summary.csv


split,row_count,label_0_count,label_1_count,label_1_ratio,timestamp_min,timestamp_max
str,u32,u32,u32,f64,datetime[μs],datetime[μs]
"""train""",3046861,3044564,2297,0.000754,2022-09-01 00:00:00,2022-09-06 13:35:00
"""val""",1015920,1014837,1083,0.001066,2022-09-06 13:36:00,2022-09-08 16:12:00
"""test""",1015564,1013767,1797,0.001769,2022-09-08 16:13:00,2022-09-18 16:18:00


## 7. time feature 포함 버전 생성
시간대, 요일 등 시간 기반 피처가 성능에 도움이 되는지 비교하기 위한 버전입니다. 최종 사용 여부는 baseline 결과를 보고 판단합니다.

In [8]:
# ============================================================
# ML-00: 현재 거래 feature
# [컬럼명 원칙] 기존 전처리파트 output 컬럼명을 그대로 유지합니다.
# ML파트 공식 표시명은 feature_catalog.csv에 alias로만 기록됩니다.
# 추가 피처(amount_received, ratio)는 기존 명명 규칙을 따릅니다.
# ============================================================

# [QA Fix] polars dt.weekday() 반환값 확인
# polars: 1=월, 2=화, ..., 6=토, 7=일 (ISO 8601)
# 코드에서 0=월~6=일로 맞추기 위해 -1을 적용합니다.
# is_weekend: 토(5) 또는 일(6) -> weekday-1 >= 5

def add_ml00_features(df):
    exprs = [
        # ── 기존 컬럼 (전처리파트 output 원본 유지) ──────────────────────────
        pl.col('amount').alias('amount__current__value'),
        safe_log1p_expr('amount').alias('amount__current__log1p'),
        pl.col('timestamp').dt.hour().cast(pl.Int16).alias('time__row__hour'),
        # dt.weekday()-1: 0=월, 1=화, 2=수, 3=목, 4=금, 5=토, 6=일
        (pl.col('timestamp').dt.weekday() - 1).cast(pl.Int16).alias('time__row__dayofweek'),
        # is_weekend: 토(5) 또는 일(6)
        ((pl.col('timestamp').dt.weekday() - 1) >= 5).cast(pl.Int8).alias('time__row__is_weekend'),
    ]

    # ── 추가 피처: amount_received (있으면 생성, 없으면 건너뜀) ──────────
    if 'amount_received' in df.columns:
        exprs += [
            pl.col('amount_received').cast(pl.Float64).alias('amount_received__current__value'),
            safe_log1p_expr('amount_received').alias('amount_received__current__log1p'),
        ]
        exprs.append(
            pl.when(
                pl.col('amount_received').is_null() |
                (pl.col('amount_received').cast(pl.Float64) == 0.0)
            ).then(None)
             .otherwise(pl.col('amount').cast(pl.Float64) / pl.col('amount_received').cast(pl.Float64))
             .alias('amount__paid_recv_ratio')
        )
    else:
        print('[INFO] amount_received 컬럼 없음 -> amount_received 계열 피처 생성 건너뜀.')

    return df.with_columns(exprs)


df_ml00_base = add_ml00_features(df)
record_memory('ml00_base_features_created', df_ml00_base)

# [QA Fix] weekday 값 범위 검증
_wd_vals = df_ml00_base['time__row__dayofweek'].drop_nulls().unique().sort().to_list()
assert all(0 <= v <= 6 for v in _wd_vals), f'time__row__dayofweek 범위 이상: {_wd_vals}'
_we_vals = set(df_ml00_base['time__row__is_weekend'].drop_nulls().unique().to_list())
assert _we_vals <= {0, 1}, f'time__row__is_weekend 값 이상: {_we_vals}'
print(f'[OK] time__row__dayofweek range: {min(_wd_vals)}~{max(_wd_vals)} (0=월,6=일)')
print(f'[OK] time__row__is_weekend values: {sorted(_we_vals)}')

_preview = [
    'tx_id', 'timestamp', 'split', 'label',
    'amount__current__value', 'amount__current__log1p',
    'time__row__hour', 'time__row__dayofweek', 'time__row__is_weekend',
    'from_bank', 'to_bank', 'payment_currency', 'receiving_currency', 'payment_format'
]
df_ml00_base.select([c for c in _preview if c in df_ml00_base.columns]).head(5)


[MEM] ml00_base_features_created | rows= 5078345 | cols= 40 | mb= 5851.21
[OK] time__row__dayofweek range: 0~6 (0=월,6=일)
[OK] time__row__is_weekend values: [0, 1]


tx_id,timestamp,split,label,amount__current__value,amount__current__log1p,time__row__hour,time__row__dayofweek,time__row__is_weekend,from_bank,to_bank,payment_currency,receiving_currency,payment_format
str,datetime[μs],str,i64,f64,f64,i16,i16,i8,str,str,str,str,str
"""0""",2022-09-01 00:00:00,"""train""",0,14675.57,9.594008,0,3,0,"""03209""","""03209""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1""",2022-09-01 00:00:00,"""train""",0,897.37,6.800582,0,3,0,"""01420""","""01420""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""10""",2022-09-01 00:00:00,"""train""",0,12971.84,9.470613,0,3,0,"""03284""","""03284""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""100""",2022-09-01 00:00:00,"""train""",0,15.69,2.81481,0,3,0,"""01601""","""01601""","""US Dollar""","""US Dollar""","""Reinvestment"""
"""1000""",2022-09-01 00:00:00,"""train""",0,924649.12,13.737171,0,3,0,"""019474""","""019474""","""US Dollar""","""US Dollar""","""Reinvestment"""


## 8. 범주형 피처 인코딩

ML파트 전달용 기본 산출물에 맞춰 categorical feature는 train split에서만 fit한 label/code 인코딩으로 추가합니다.

- 기존 feature 계산 로직과 `used_in_ml=True` 목록은 유지합니다.
- 최종 parquet에는 raw CSV 원본 컬럼을 함께 보존합니다.
- one-hot/hybrid 비교용 산출물은 만들지 않습니다.
- validation/test에서 train에 없던 category는 `-1`로 처리합니다.
- 원문 계좌 식별자, label/split/timestamp, pattern/laundering 관련 정보는 모델 입력 feature에서 제외합니다.

In [9]:
# ============================================================
# 범주형 인코딩
# - fit은 train 데이터만 사용한다.
# - val/test 신규 category는 label/code -1로 처리한다.
# - ML파트 전달용 최종 산출물은 기존 구조 + label/code encoded columns만 생성한다.
# ============================================================

CATEGORICAL_RAW_COLS = [
    "from_bank",
    "to_bank",
    "payment_currency",
    "receiving_currency",
    "payment_format",
]
print("CATEGORICAL_RAW_COLS:", CATEGORICAL_RAW_COLS)

CAT_FEATURE_MAP = {
    raw_col: "cat__{}__code".format(raw_col)
    for raw_col in CATEGORICAL_RAW_COLS
}


def _normalized_string_expr(col):
    return pl.col(col).cast(pl.Utf8).fill_null("unknown")


def fit_label_maps_train_only(df, cols):
    train_df = df.filter(pl.col("split") == "train")
    if train_df.height == 0:
        raise ValueError("train split이 비어 있어 categorical encoder를 fit할 수 없습니다.")

    encoders = {}

    for col in cols:
        vc = (
            train_df
            .select(_normalized_string_expr(col).alias(col))
            .group_by(col)
            .agg(pl.len().alias("train_count"))
            .sort(col)
        )

        values = [str(v) for v in vc[col].to_list()]
        counts = [int(v) for v in vc["train_count"].to_list()]
        label_map = {category: i for i, category in enumerate(values)}
        count_map = dict(zip(values, counts))
        ratio_map = {
            category: count_map[category] / train_df.height
            for category in values
        }

        encoders[col] = {
            "label_map": label_map,
            "train_count": count_map,
            "train_ratio": ratio_map,
            "train_values": set(values),
        }
        print("[fit:label_code]", col, "n_unique_train=", len(values))

    return encoders


def apply_label_maps(df, encoders):
    out = df

    for raw_col, encoder in encoders.items():
        feature_col = CAT_FEATURE_MAP[raw_col]
        mapping = encoder["label_map"]

        map_df = pl.DataFrame({
            raw_col: list(mapping.keys()),
            feature_col: list(mapping.values()),
        }).with_columns([
            pl.col(raw_col).cast(pl.Utf8),
            pl.col(feature_col).cast(pl.Int32),
        ])

        out = (
            out.with_columns(_normalized_string_expr(raw_col).alias(raw_col))
            .join(map_df, on=raw_col, how="left")
            .with_columns(pl.col(feature_col).fill_null(-1).cast(pl.Int32))
        )

    return out


# category encoder는 train split만 기준으로 fit합니다.
CATEGORY_ENCODERS = fit_label_maps_train_only(df_ml00_base, CATEGORICAL_RAW_COLS)
df_ml00_base = apply_label_maps(df_ml00_base, CATEGORY_ENCODERS)

CAT_LABEL_FEATURE_COLS = [CAT_FEATURE_MAP[c] for c in CATEGORICAL_RAW_COLS]

print("label/code categorical features:", len(CAT_LABEL_FEATURE_COLS))

record_memory("categorical_encoded_train_only", df_ml00_base)

show_cols = ["split"]
for raw_col in CATEGORICAL_RAW_COLS:
    show_cols += [raw_col, CAT_FEATURE_MAP[raw_col]]

df_ml00_base.select(show_cols).head(5) if show_cols != ["split"] else df_ml00_base.select(show_cols).head(5)


# ============================================================
# Pair Encoding (optional candidate)
# 컬럼명은 기존 cat__*__code 규칙 동일하게 적용
# used_in_ml=True/False는 ml_feature_columns.csv로 결정
# ============================================================

PAIR_ENCODING_DEFS = {
    'cat__currency_pair__code': ('payment_currency', 'receiving_currency'),
    'cat__bank_pair__code':     ('from_bank', 'to_bank'),
}

def _pair_key(pair_col): return f'__praw__{pair_col}'

def add_raw_pair_cols(df):
    exprs = []
    for pair_col, (col_a, col_b) in PAIR_ENCODING_DEFS.items():
        if col_a in df.columns and col_b in df.columns:
            exprs.append(
                (pl.col(col_a).cast(pl.Utf8).fill_null('unknown') + pl.lit('||') +
                 pl.col(col_b).cast(pl.Utf8).fill_null('unknown')).alias(_pair_key(pair_col))
            )
        else:
            print(f'[SKIP] pair {pair_col}: 원본 컬럼 없음 ({col_a}, {col_b})')
    return df.with_columns(exprs) if exprs else df

def fit_pair_maps(df, pair_defs):
    train_df = df.filter(pl.col('split') == 'train')
    encoders = {}
    for pair_col, (col_a, col_b) in pair_defs.items():
        raw_col = _pair_key(pair_col)
        if raw_col not in df.columns: continue
        vals = (train_df.select(pl.col(raw_col)).group_by(raw_col)
                        .agg(pl.len()).sort(raw_col)[raw_col].to_list())
        encoders[pair_col] = {'map': {v:i for i,v in enumerate(vals)}, 'raw_col': raw_col}
        print(f'[fit:pair_code] {pair_col} n_train={len(vals)}')
    return encoders

def apply_pair_maps(df, encoders):
    out = df
    for pair_col, enc in encoders.items():
        raw_col = enc['raw_col']
        m = enc['map']
        map_df = pl.DataFrame({raw_col: list(m.keys()), pair_col: list(m.values())})
        map_df = map_df.with_columns([pl.col(raw_col).cast(pl.Utf8), pl.col(pair_col).cast(pl.Int32)])
        out = (out.join(map_df, on=raw_col, how='left')
                  .with_columns(pl.col(pair_col).fill_null(-1).cast(pl.Int32)))
    return out

df_ml00_base = add_raw_pair_cols(df_ml00_base)
PAIR_ENCODERS = fit_pair_maps(df_ml00_base, PAIR_ENCODING_DEFS)
df_ml00_base = apply_pair_maps(df_ml00_base, PAIR_ENCODERS)

PAIR_FEATURE_COLS = list(PAIR_ENCODERS.keys())
print('pair feature cols (candidate):', PAIR_FEATURE_COLS)
print('-> used_in_ml 기본값: False (ml_feature_columns.csv에서 True로 바꿔야 활성화)')

record_memory('pair_encoded', df_ml00_base)

CATEGORICAL_RAW_COLS: ['from_bank', 'to_bank', 'payment_currency', 'receiving_currency', 'payment_format']
[fit:label_code] from_bank n_unique_train= 30528
[fit:label_code] to_bank n_unique_train= 15850
[fit:label_code] payment_currency n_unique_train= 15
[fit:label_code] receiving_currency n_unique_train= 15
[fit:label_code] payment_format n_unique_train= 7
label/code categorical features: 5
[MEM] categorical_encoded_train_only | rows= 5078345 | cols= 45 | mb= 5871.99
[fit:pair_code] cat__currency_pair__code n_train=208
[fit:pair_code] cat__bank_pair__code n_train=262735
pair feature cols (candidate): ['cat__currency_pair__code', 'cat__bank_pair__code']
-> used_in_ml 기본값: False (ml_feature_columns.csv에서 True로 바꿔야 활성화)
[MEM] pair_encoded | rows= 5078345 | cols= 49 | mb= 6084.91


## 11. 모델 입력 feature 목록 정의

`ML00_FEATURE_COLS` 등 각 실험의 feature 목록을 확정합니다.  
pair / amount_received 후보 컬럼은 parquet에는 포함하되 `used_in_ml=False` 기본값으로 관리합니다.

In [10]:
# ============================================================
# 모델 입력 feature 목록
# [원칙] 기존 전처리파트 output 컬럼명 유지.
#          amount_received 계열 → add_candidate (used_in_ml=False)
#          pair feature → candidate (used_in_ml=False, parquet에 포함)
# ============================================================

META_COLS = ['tx_id','timestamp','split','label','sender_account','receiver_account']

# ── ml_exp00 기본 feature (used_in_ml=True 대상) ──────────────────────
AMOUNT_BASE_COLS = ['amount__current__value', 'amount__current__log1p']
TIME_FEATURE_COLS = ['time__row__hour','time__row__dayofweek','time__row__is_weekend']
CAT_FEATURE_COLS_ALL = CAT_LABEL_FEATURE_COLS  # 5개 기본

ML00_FEATURE_COLS         = AMOUNT_BASE_COLS + CAT_FEATURE_COLS_ALL + TIME_FEATURE_COLS
ML00_NO_TIME_FEATURE_COLS = AMOUNT_BASE_COLS + CAT_FEATURE_COLS_ALL

# ── amount_received 계열: add_candidate (parquet에 존재 시 활성화 가능) ─
# ML00_FEATURE_COLS에는 포함하지 않음. used_in_ml=False 기본값.
AMOUNT_RECV_CANDIDATE_COLS = [
    c for c in ['amount_received__current__value',
                'amount_received__current__log1p',
                'amount__paid_recv_ratio']
    if c in df_ml00_base.columns
]
if AMOUNT_RECV_CANDIDATE_COLS:
    print('[INFO] amount_received 계열 parquet에 포함 (candidate, used_in_ml=False):', AMOUNT_RECV_CANDIDATE_COLS)
else:
    print('[INFO] amount_received 컬럼 없음 -> 관련 후보 피처 없음')

# ── pair feature: candidate (parquet에 포함, used_in_ml=False) ───────
PAIR_COLS_IN_PARQUET = [c for c in PAIR_FEATURE_COLS if c in df_ml00_base.columns]
if len(PAIR_COLS_IN_PARQUET) < len(PAIR_FEATURE_COLS):
    print(f'[WARN] pair col 미생성: {[c for c in PAIR_FEATURE_COLS if c not in df_ml00_base.columns]}')

FEATURE_SET_REGISTRY = {
    'ml_exp00':         ML00_FEATURE_COLS,
    'ml_exp00_no_time': ML00_NO_TIME_FEATURE_COLS,
}

print(f'ML00 feature cols    : {len(ML00_FEATURE_COLS)}')
print(f'ML00 no-time cols    : {len(ML00_NO_TIME_FEATURE_COLS)}')
print(f'amount_recv candidate: {AMOUNT_RECV_CANDIDATE_COLS}')
print(f'pair candidate       : {PAIR_COLS_IN_PARQUET} (parquet 포함, used_in_ml=False)')

for name, cols in FEATURE_SET_REGISTRY.items():
    bad = [c for c in cols if c not in df_ml00_base.columns]
    if bad: print(f'[WARN] {name} missing: {bad}')


[INFO] amount_received 계열 parquet에 포함 (candidate, used_in_ml=False): ['amount_received__current__value', 'amount_received__current__log1p', 'amount__paid_recv_ratio']
ML00 feature cols    : 10
ML00 no-time cols    : 7
amount_recv candidate: ['amount_received__current__value', 'amount_received__current__log1p', 'amount__paid_recv_ratio']
pair candidate       : ['cat__currency_pair__code', 'cat__bank_pair__code'] (parquet 포함, used_in_ml=False)


## 12. 최종 ML 데이터셋 구성

`ml_exp00`, `ml_exp00_no_time`을 저장 대상으로 구성합니다. history feature(`ml_exp01`)는 메모리에 계산하되 parquet로 저장하지 않습니다.  
raw CSV 원본 컬럼과 pair / amount_received 후보 컬럼은 parquet에 포함하되 ML feature list에는 넣지 않습니다.

In [11]:
# ============================================================
# 최종 ML 데이터셋
# [QA Fix] pair / amount_received 후보 feature는 parquet에 포함
#          모델 사용 여부는 ml_feature_columns.csv의 used_in_ml로 제어
# ============================================================
def select_existing_cols(df, cols):
    out = []
    seen = set()
    for col in cols:
        if col in df.columns and col not in seen:
            out.append(col)
            seen.add(col)
    return out

RAW_ORIGINAL_COLS = [
    'Timestamp',
    'From Bank',
    'Account',
    'To Bank',
    'Account.1',
    'Amount Received',
    'Receiving Currency',
    'Amount Paid',
    'Payment Currency',
    'Payment Format',
    'Is Laundering',
]
META_KEEP_COLS = select_existing_cols(df_ml00_base, META_COLS)
RAW_ORIGINAL_KEEP_COLS = select_existing_cols(df_ml00_base, RAW_ORIGINAL_COLS)
_missing_raw_original_cols = [c for c in RAW_ORIGINAL_COLS if c not in df_ml00_base.columns]
if _missing_raw_original_cols:
    print(f'[WARN] raw original cols missing from clean_base: {_missing_raw_original_cols}')
if not RAW_ORIGINAL_KEEP_COLS:
    raise ValueError('[RAW ORIGINAL] clean_base에서 raw CSV 원본 컬럼을 찾지 못했습니다.')
print('raw original cols in parquet:', RAW_ORIGINAL_KEEP_COLS)
# amount_received 계열은 parquet에 포함하되 ML feature set(ML00_FEATURE_COLS)에는 없음
_amount_recv_keep = select_existing_cols(df_ml00_base, AMOUNT_RECV_CANDIDATE_COLS)
# pair col은 parquet에 포함하되 ML feature set(ML00_FEATURE_COLS)에는 없음
_pair_keep = select_existing_cols(df_ml00_base, PAIR_COLS_IN_PARQUET)
_ml_exp00_cols = select_existing_cols(
    df_ml00_base,
    META_KEEP_COLS + RAW_ORIGINAL_KEEP_COLS + ML00_FEATURE_COLS + _amount_recv_keep + _pair_keep,
)
_ml_exp00_no_time_cols = select_existing_cols(
    df_ml00_base,
    META_KEEP_COLS + RAW_ORIGINAL_KEEP_COLS + ML00_NO_TIME_FEATURE_COLS + _amount_recv_keep + _pair_keep,
)
ml_exp00 = df_ml00_base.select(_ml_exp00_cols)
ml_exp00_no_time = df_ml00_base.select(_ml_exp00_no_time_cols)
record_memory('ml_exp00_ready', ml_exp00)
record_memory('ml_exp00_no_time_ready', ml_exp00_no_time)
for name, data in [
    ('ml_exp00', ml_exp00), ('ml_exp00_no_time', ml_exp00_no_time),
]:
    print_shape(name, data)
    if _amount_recv_keep:
        _rc = [c for c in _amount_recv_keep if c in data.columns]
        print(f'  amount_received cols in parquet: {_rc}')
    if RAW_ORIGINAL_KEEP_COLS:
        _raw = [c for c in RAW_ORIGINAL_KEEP_COLS if c in data.columns]
        print(f'  raw original cols in parquet: {_raw}')
    if _pair_keep:
        _pc = [c for c in _pair_keep if c in data.columns]
        print(f'  pair cols in parquet: {_pc}')

raw original cols in parquet: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
[MEM] ml_exp00_ready | rows= 5078345 | cols= 32 | mb= 6076.1
[MEM] ml_exp00_no_time_ready | rows= 5078345 | cols= 29 | mb= 6076.1
ml_exp00: rows=5,078,345, cols=32
  amount_received cols in parquet: ['amount_received__current__value', 'amount_received__current__log1p', 'amount__paid_recv_ratio']
  raw original cols in parquet: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']
  pair cols in parquet: ['cat__currency_pair__code', 'cat__bank_pair__code']
ml_exp00_no_time: rows=5,078,345, cols=29
  amount_received cols in parquet: ['amount_received__current__value', 'amount_received__current__log1p', 'amount__paid_recv_ratio']
  raw original cols in parquet: ['Timestamp

## 13. feature catalog / ml_feature_columns

각 피처의 출처, 계산 방식, leakage 규칙, alias를 기록합니다.  
`ml_feature_columns.csv`는 `ml_io.load_feature_columns()`의 입력 기준입니다.

In [12]:
# ============================================================
# feature_catalog.csv
# [컬럼명 원칙] parquet_column_name = 기존 전처리파트 컬럼명 유지
# ============================================================

def make_feature_catalog():
    records = []

    def add(cur, desc, src, calc, dtype, unit, tw, miss, leak, uml, notes, owner, status='selected'):
        records.append({
            'feature_name': cur,
            'description': desc, 'source_data': src,
            'calculation_method': calc, 'data_type': dtype,
            'unit': unit, 'time_window': tw,
            'missing_policy': miss, 'leakage_risk': leak,
            'used_in_ml': uml, 'notes': notes,
            'owner_part': owner, 'selection_status': status,
        })

    # 금액 (기존 컬럼명 유지)
    add('amount__current__value',
        '현재 거래 지급 금액(amount 컬럼)','amount','identity','numeric','원','현재거래','비null필수','current row only',True,'기존 전처리파트 컬럼','PART1')
    add('amount__current__log1p',
        '지급 금액 log1p','amount','log1p','numeric','log(원+1)','현재거래','음수->null','current row only',True,'기존 전처리파트 컬럼','PART1')
    add('amount_received__current__value',
        '현재 거래 수취 금액','amount_received','identity','numeric','원','현재거래','비null필수','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')
    add('amount_received__current__log1p',
        '수취 금액 log1p','amount_received','log1p','numeric','log(원+1)','현재거래','음수->null','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')
    add('amount__paid_recv_ratio',
        '지급/수취 금액 비율','amount,amount_received','paid/received','numeric','무차원','현재거래','received=0->null','current row only',False,
        'add_candidate: 기본 used_in_ml=False. parquet에 포함됨. ml_feature_columns.csv에서 True로 변경 시 활성화 가능','PART1','add_candidate')

    # 시간 (기존 컬럼명 유지)
    add('time__row__hour',
        '거래 시간(0-23)','timestamp','dt.hour()','numeric/int','시','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__dayofweek',
        '거래 요일(0=월~6=일)','timestamp','dt.weekday()','numeric/int','요일코드','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__is_weekend',
        '주말여부(1=주말)','timestamp','weekday>=6->1','numeric/int','0/1','현재거래','non-null','current ts only',True,'기존 전처리파트 컬럼','PART1')
    add('time__row__elapsed_h',
        '데이터시작 후 경과시간','timestamp','(ts-min)/3600','numeric','시간(h)','전체데이터','계산가능','전체기간위치->누수위험',False,'optional. 누수위험. used_in_ml=False','PART1','optional_leakage_risk')

    # categorical 5개 (기존 컬럼명 유지)
    for raw_col, feat_col in CAT_FEATURE_MAP.items():
        add(feat_col, f'{raw_col} train-only code', raw_col, 'train-only label code','numeric/int','코드(순서없음)','현재거래','unseen->-1','train only fit',True,'기존 전처리파트 컬럼. code값에 순서 의미 없음','PART1')

    # pair feature (candidate, 기존 naming 규칙 동일)
    for pair_col, (col_a, col_b) in PAIR_ENCODING_DEFS.items():
        add(pair_col, f'{col_a}+{col_b} pair code',
            f'{col_a},{col_b}', 'colA||colB train-only code','numeric/int','코드(순서없음)','현재거래','unseen pair->-1','train only fit',False,
            'candidate. 성능비교후 used_in_ml=True로 활성화 가능','PART2','candidate')


    # 원시 원본 컬럼 — WOE/IV 계산은 되지만 모델 입력에서 제외
    for raw_col, desc in [
        ('Amount Paid',          '원시 지급 금액 컬럼 (amount__current__value 의 원본)'),
        ('Amount Received',      '원시 수취 금액 컬럼 (amount_received__current__value 의 원본)'),
        ('From Bank',            '원시 송신 은행 컬럼 (cat__from_bank__code 의 원본)'),
        ('To Bank',              '원시 수신 은행 컬럼 (cat__to_bank__code 의 원본)'),
        ('Payment Currency',     '원시 지급 통화 컬럼 (cat__payment_currency__code 의 원본)'),
        ('Receiving Currency',   '원시 수취 통화 컬럼 (cat__receiving_currency__code 의 원본)'),
        ('Payment Format',       '원시 결제 방식 컬럼 (cat__payment_format__code 의 원본)'),
    ]:
        add(raw_col, desc, raw_col, 'raw', 'raw', '-', '현재거래', '-', 'raw column',
            False, '원시 컬럼. code 피처로 대체됨. used_in_ml=False', 'PART1', 'excluded_raw')

    return pd.DataFrame(records)


feature_catalog_df = make_feature_catalog()
catalog_path = CONFIG['DRIVE_OUTPUT_DIR'] / 'feature_catalog.csv'
feature_catalog_df.to_csv(catalog_path, index=False, encoding='utf-8-sig')
print('saved:', catalog_path)

# WOE/IV 결과 폴더에도 저장
_leaderboard_dir = DRIVE_BASE_DIR / 'leader_board'
for _exp_name in ['ml_exp00', 'ml_exp00_no_time']:
    _woe_dir = _leaderboard_dir / _exp_name
    _woe_dir.mkdir(parents=True, exist_ok=True)
    _woe_catalog = _woe_dir / 'feature_catalog.csv'
    feature_catalog_df.to_csv(_woe_catalog, index=False, encoding='utf-8-sig')
    print('saved:', _woe_catalog)

display(feature_catalog_df[['feature_name','description','used_in_ml','selection_status']].head(25))

saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/feature_catalog.csv
saved: /content/drive/MyDrive/Graph_AML/leader_board/ml_exp00/feature_catalog.csv
saved: /content/drive/MyDrive/Graph_AML/leader_board/ml_exp00_no_time/feature_catalog.csv


,feature_name,description,used_in_ml,selection_status
0,amount__current__value,현재 거래 지급 금액(amount 컬럼),True,selected
1,amount__current__log1p,지급 금액 log1p,True,selected
2,amount_received__current__value,현재 거래 수취 금액,False,add_candidate
3,amount_received__current__log1p,수취 금액 log1p,False,add_candidate
4,amount__paid_recv_ratio,지급/수취 금액 비율,False,add_candidate
5,time__row__hour,거래 시간(0-23),True,selected
6,time__row__dayofweek,거래 요일(0=월~6=일),True,selected
7,time__row__is_weekend,주말여부(1=주말),True,selected
8,time__row__elapsed_h,데이터시작 후 경과시간,False,optional_leakage_risk
9,cat__from_bank__code,from_bank train-only code,True,selected


## 14. leakage check / parquet 저장 / schema 호환성 검증

P0 leakage 점검 → 이상 없으면 parquet 저장 → schema 호환성 검증 순서로 실행합니다.  
`ml_feature_columns.csv`의 `used_in_ml=True` 컬럼이 저장된 parquet에 모두 존재하는지 확인합니다.

In [13]:
# ============================================================
# ml_feature_columns.csv  (A안: ml_exp00 전용 feature가 기본 used_in_ml=True)
# history(ml_exp01) / amount_received 계열 / pair는 False 기본값
# LR baseline은 이 CSV를 읽으므로 ml_exp00 parquet과 충돌 없음
# ============================================================

def infer_feature_group(col):
    if col.startswith('amount__') or col.startswith('amount_received__'): return 'amount'
    if col.startswith('time__'):     return 'time'
    if col.startswith('cat__'):      return 'categorical_label_code'
    if col.startswith('timehist__'): return 'history'
    if col.startswith('recency__'):  return 'recency'
    if col.startswith('flag__'):     return 'flag'
    return 'unknown'

def infer_leakage_risk(col):
    if 'elapsed' in col:             return '누수위험_optional'
    if any(col.startswith(p) for p in ['timehist__','recency__','flag__']): return 'low_if_ts_lt_current'
    if col.startswith('cat__'):       return 'low_if_train_only_mapping'
    if col.startswith('time__'):      return 'low_current_ts_only'
    return 'low_current_row_only'

# 카탈로그에 포함할 모든 컬럼 목록
_RECV_SET  = {'amount_received__current__value','amount_received__current__log1p','amount__paid_recv_ratio'}
_PAIR_SET  = set(PAIR_FEATURE_COLS)
_ML00_SET  = set(ML00_FEATURE_COLS)  # 기본 used_in_ml=True 대상

def make_ml_feature_columns():
    all_cols = sorted(
        set(c for cols in FEATURE_SET_REGISTRY.values() for c in cols)
        | set(PAIR_COLS_IN_PARQUET)
        | set(AMOUNT_RECV_CANDIDATE_COLS)
    )
    records = []
    for col in all_cols:
        # used_in_ml 결정 (A안: ml_exp00 base만 True)
        if col in _ML00_SET:
            used, excl = True, ''
        elif col in _RECV_SET:
            used, excl = False, 'add_candidate: 기본 False. parquet에 있으면 True로 변경 가능'
        elif col in _PAIR_SET:
            used, excl = False, 'candidate: 성능비교 후 True로 변경 (parquet에 포함됨)'
        elif 'elapsed' in col:
            used, excl = False, '시간전체위치 누수위험 optional'
        else:
            used, excl = False, '미분류'

        # 어느 실험에 속하는지
        in_exps = ';'.join(n for n,cs in FEATURE_SET_REGISTRY.items() if col in cs)

        records.append({
            'column_name':        col,
            'used_in_ml':         used,
            'target_experiment':  'ml_exp00' if col in _ML00_SET else 'candidate',
            'used_in_experiments': in_exps,
            'feature_group':      infer_feature_group(col),
            'dtype':              safe_schema_get(df_ml00_base.schema, col, 'unknown'),
            'excluded_reason':    excl,
            'leakage_risk':       infer_leakage_risk(col),
            'selection_note':     'candidate (parquet 포함)' if col in _PAIR_SET else '',
        })

    # meta/제외 컬럼
    excluded_cols = select_existing_cols(
        df_ml00_base,
        ['tx_id','timestamp','split','label','sender_account','receiver_account',
         'from_bank','to_bank','payment_currency','receiving_currency','payment_format']
        + RAW_ORIGINAL_KEEP_COLS,
    )
    for col in excluded_cols:
        if col in df_ml00_base.columns:
            records.append({
                'column_name':col,'used_in_ml':False,'target_experiment':'excluded',
                'used_in_experiments':'','feature_group':'meta_or_raw',
                'dtype':safe_schema_get(df_ml00_base.schema,col),
                'excluded_reason':'model input 제외','leakage_risk':'excluded','selection_note':''
            })
    return pd.DataFrame(records)

ml_feature_columns_df = make_ml_feature_columns()
ml_feature_columns_path = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_feature_columns.csv'
ml_feature_columns_df.to_csv(ml_feature_columns_path, index=False, encoding='utf-8-sig')
print('saved:', ml_feature_columns_path)
_n_true = ml_feature_columns_df['used_in_ml'].sum()
_n_total = len(ml_feature_columns_df)
print(f'used_in_ml=True: {_n_true} / total: {_n_total}')
print(f'True 컬럼은 모두 ml_exp00 parquet에 존재해야 합니다.')
display(ml_feature_columns_df[['column_name','used_in_ml','target_experiment',
                                'feature_group','excluded_reason']].head(40))

saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_feature_columns.csv
used_in_ml=True: 10 / total: 37
True 컬럼은 모두 ml_exp00 parquet에 존재해야 합니다.


,column_name,used_in_ml,target_experiment,feature_group,excluded_reason
0,amount__current__log1p,True,ml_exp00,amount,
1,amount__current__value,True,ml_exp00,amount,
2,amount__paid_recv_ratio,False,candidate,amount,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
3,amount_received__current__log1p,False,candidate,amount,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
4,amount_received__current__value,False,candidate,amount,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
5,cat__bank_pair__code,False,candidate,categorical_label_code,candidate: 성능비교 후 True로 변경 (parquet에 포함됨)
6,cat__currency_pair__code,False,candidate,categorical_label_code,candidate: 성능비교 후 True로 변경 (parquet에 포함됨)
7,cat__from_bank__code,True,ml_exp00,categorical_label_code,
8,cat__payment_currency__code,True,ml_exp00,categorical_label_code,
9,cat__payment_format__code,True,ml_exp00,categorical_label_code,


## 15. 검증 및 저장 (leakage check → parquet → schema 검증)

순서: leakage check → parquet 저장 → schema 호환성 검증

In [14]:
# ============================================================
# leakage check
# P0 실패 시 파일 저장 중단
# ============================================================

LEAKAGE_RESULTS = []

# leakage check 결과를 같은 형식으로 모읍니다.
def add_check_result(check_name, status, severity, details, n_failed_rows=0):
    LEAKAGE_RESULTS.append({
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": details,
        "n_failed_rows": int(n_failed_rows),
    })

    mark = "PASS" if status == "PASS" else "FAIL"
    print("[{}] [{}] {} | failed={} | {}".format(
        mark, severity, check_name, n_failed_rows, details
    ))


def _get_split_ts(summary_pd, split_name, col):
    sub = summary_pd.loc[summary_pd["split"] == split_name, col]
    if sub.empty:
        raise ValueError("split '{}' 가 비어 있습니다.".format(split_name))
    return pd.to_datetime(sub.iloc[0])


def validate_time_split(df):
    s = make_split_summary(df).to_pandas()

    train_max = _get_split_ts(s, "train", "timestamp_max")
    val_min = _get_split_ts(s, "val", "timestamp_min")
    val_max = _get_split_ts(s, "val", "timestamp_max")
    test_min = _get_split_ts(s, "test", "timestamp_min")

    ok = train_max <= val_min and val_max <= test_min

    add_check_result(
        "validate_time_split",
        "PASS" if ok else "FAIL",
        "P0",
        "train_max={}, val_min={}, val_max={}, test_min={}".format(
            train_max, val_min, val_max, test_min
        ),
        0 if ok else df.height,
    )




def validate_no_forbidden_ml_columns(feature_cols, dataset_name):
    exact_forbidden = {x.lower() for x in CONFIG["EXACT_FORBIDDEN_MODEL_COLUMNS"]}
    substring_forbidden = {x.lower() for x in CONFIG["SUBSTRING_FORBIDDEN_MODEL_TOKENS"]}

    failed = []

    for col in feature_cols:
        lower_col = col.lower()

        # 파생 시간 피처와 recency 피처는 허용
        # 원본 timestamp 컬럼 자체는 exact forbidden에서 차단
        if col.startswith("time__row__") or col.startswith("recency__"):
            continue

        if lower_col in exact_forbidden:
            failed.append(col)
            continue

        for token in substring_forbidden:
            if token in lower_col:
                failed.append(col)
                break

    failed = sorted(set(failed))

    add_check_result(
        "validate_no_forbidden_ml_columns__{}".format(dataset_name),
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "forbidden columns: {}".format(failed[:20]),
        len(failed),
    )




def validate_categorical_unknown_handling(df_with_features):
    failed = []

    for raw_col in CATEGORICAL_RAW_COLS:
        train_values = CATEGORY_ENCODERS[raw_col]["train_values"]
        code_col = CAT_FEATURE_MAP[raw_col]

        for split_name in ["val", "test"]:
            unknown_rows = df_with_features.filter(
                (pl.col("split") == split_name)
                & (~_normalized_string_expr(raw_col).is_in(list(train_values)))
            )

            if unknown_rows.height > 0:
                wrong_code = unknown_rows.filter(pl.col(code_col) != -1).height
                if wrong_code > 0:
                    failed.append({
                        "raw_column": raw_col,
                        "split": split_name,
                        "issue": "unseen_category_code_not_minus_one",
                        "n_failed_rows": wrong_code,
                    })

    add_check_result(
        "validate_categorical_unknown_handling",
        "PASS" if len(failed) == 0 else "FAIL",
        "P0",
        "first_failures={}".format(failed[:5]),
        sum(x["n_failed_rows"] for x in failed) if failed else 0,
    )






def run_all_ml_validations():
    global LEAKAGE_RESULTS
    LEAKAGE_RESULTS = []

    _required_runtime_vars = [
        "df",
        "df_ml00_base",
        "FEATURE_SET_REGISTRY",
    ]
    _missing = [v for v in _required_runtime_vars if v not in globals()]
    if _missing:
        raise RuntimeError(
            "[RUN ORDER ERROR] 앞 셀이 실행되지 않아 필수 변수가 없습니다: "
            + ", ".join(_missing)
            + "\n런타임 재시작 후 전체 실행하거나, Cell 18~24를 먼저 실행한 뒤 leakage check를 실행하세요."
        )

    validate_time_split(df)

    for dataset_name, feature_cols in FEATURE_SET_REGISTRY.items():
        validate_no_forbidden_ml_columns(feature_cols, dataset_name)

    validate_categorical_unknown_handling(df_ml00_base)


    leakage_df = pd.DataFrame(LEAKAGE_RESULTS)
    leakage_path = CONFIG["VALIDATION_DIR"] / "leakage_check.csv"
    leakage_df.to_csv(leakage_path, index=False, encoding="utf-8-sig")
    print("saved:", leakage_path)

    p0_fail = leakage_df[
        (leakage_df["severity"] == "P0")
        & (leakage_df["status"] != "PASS")
    ]

    if len(p0_fail) > 0:
        display(p0_fail)
        raise AssertionError("P0 leakage check failed. 파일 저장을 중단합니다.")

    return leakage_df


leakage_check = run_all_ml_validations()
leakage_check

[PASS] [P0] validate_time_split | failed=0 | train_max=2022-09-06 13:35:00, val_min=2022-09-06 13:36:00, val_max=2022-09-08 16:12:00, test_min=2022-09-08 16:13:00
[PASS] [P0] validate_no_forbidden_ml_columns__ml_exp00 | failed=0 | forbidden columns: []
[PASS] [P0] validate_no_forbidden_ml_columns__ml_exp00_no_time | failed=0 | forbidden columns: []
[PASS] [P0] validate_categorical_unknown_handling | failed=0 | first_failures=[]
saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/validation/leakage_check.csv


,check_name,status,severity,details,n_failed_rows
0,validate_time_split,PASS,P0,"train_max=2022-09-06 13:35:00, val_min=2022-09...",0
1,validate_no_forbidden_ml_columns__ml_exp00,PASS,P0,forbidden columns: [],0
2,validate_no_forbidden_ml_columns__ml_exp00_no_...,PASS,P0,forbidden columns: [],0
3,validate_categorical_unknown_handling,PASS,P0,first_failures=[],0


In [15]:
# ============================================================
# parquet 저장 — 단일 파일 전달 기본
#
# ML파트 피드백 반영:
#   - train/val/test를 별도 파일로 나누지 않음
#   - 하나의 parquet 안에 split 컬럼을 보존
#   - ML파트 코드는 split 컬럼을 읽어 train/val/test를 분리하면 됨
#
# 필요 시 아래 옵션을 True로 바꾸면 구버전 split 파일도 추가 저장 가능
# ============================================================

SAVE_SPLIT_PARQUETS = False   # 기본: False. train/val/test 분리 파일 생성 안 함
SAVE_XY_VARIANTS    = False   # 기본: False. Xy 중복 파일 생성 안 함


def save_parquet(df, filename):
    path = CONFIG['DRIVE_OUTPUT_DIR'] / filename
    df.write_parquet(path)
    print('saved:', path, '| rows=', df.height, '| cols=', len(df.columns))
    return path

# 기본 전달 파일: 실험별 단일 parquet 1개씩만 저장
DATASET_OBJECTS = {
    'ml_exp00': ml_exp00,
    'ml_exp00_no_time': ml_exp00_no_time,
}

# 선택 옵션: 기존 Xy 파일도 필요할 때만 추가
if SAVE_XY_VARIANTS:
    DATASET_OBJECTS.update({
        'ml_exp00_Xy': ml_exp00_Xy,
        'ml_exp00_no_time_Xy': ml_exp00_no_time_Xy,
    })

# 단일 parquet 저장 + split 컬럼 검증
for dataset_name, dataset_df in DATASET_OBJECTS.items():
    if 'split' not in dataset_df.columns:
        raise ValueError(f'[{dataset_name}] split 컬럼이 없습니다. 단일 파일 전달 불가')
    vals = set(dataset_df['split'].unique().to_list())
    if vals != {'train', 'val', 'test'}:
        raise ValueError(f'[{dataset_name}] split 값 이상: {sorted(vals)}')
    save_parquet(dataset_df, f'{dataset_name}.parquet')

# 구버전 호환용 split별 파일은 옵션이 켜진 경우에만 저장
if SAVE_SPLIT_PARQUETS:
    for dataset_name, dataset_df in DATASET_OBJECTS.items():
        for split_name in ['train', 'val', 'test']:
            save_parquet(
                dataset_df.filter(pl.col('split') == split_name),
                f'{dataset_name}_{split_name}.parquet',
            )
    print('[INFO] SAVE_SPLIT_PARQUETS=True → 구버전 train/val/test 분리 파일도 저장했습니다.')
else:
    print('[INFO] 단일 파일 전달: train/val/test 분리 parquet는 생성하지 않습니다.')

record_memory('parquet_saved_single_file_output', ml_exp00)


saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_exp00.parquet | rows= 5078345 | cols= 32
saved: /content/drive/MyDrive/Graph_AML/data/processed/ml_features/ml_exp00_no_time.parquet | rows= 5078345 | cols= 29
[INFO] 단일 파일 전달: train/val/test 분리 parquet는 생성하지 않습니다.
[MEM] parquet_saved_single_file_output | rows= 5078345 | cols= 32 | mb= 5072.38


## schema 호환성 검증

`ml_feature_columns.csv`의 `used_in_ml=True` 컬럼이 저장된 parquet에 모두 존재하는지 확인합니다.  
`ml_io.load_split` smoke test도 함께 실행합니다.

In [16]:
# ============================================================
# schema 호환성 검증 — 단일 파일 전달 기준
# - split별 parquet가 아니라 단일 parquet 안의 split 컬럼을 검증
# - pyarrow로 schema/dtype 직접 확인
# - label 분포는 split별 전체 기준으로 확인
# ============================================================

import pyarrow.parquet as _pq
import numpy as np

_feat_csv_path = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_feature_columns.csv'
_ml_cols_df = pd.read_csv(_feat_csv_path, encoding='utf-8-sig')
_true_cols = _ml_cols_df.loc[_ml_cols_df['used_in_ml'] == True, 'column_name'].tolist()
print(f'used_in_ml=True 컬럼: {len(_true_cols)}개')

_EXPERIMENT_CHECKS = [
    ('ml_exp00',         ML00_FEATURE_COLS),
    ('ml_exp00_no_time', ML00_NO_TIME_FEATURE_COLS),
]
_compat_errors = []

for exp_name, feat_list in _EXPERIMENT_CHECKS:
    _pq_path = CONFIG['DRIVE_OUTPUT_DIR'] / f'{exp_name}.parquet'
    if not _pq_path.exists():
        _compat_errors.append(f'[{exp_name}] 단일 parquet 없음: {_pq_path}')
        continue
    try:
        _pf = _pq.ParquetFile(_pq_path)
        _schema = _pf.schema_arrow
        _schema_names = set(_schema.names)

        _required = set(feat_list) | {'split', 'label'}
        _missing = [c for c in _required if c not in _schema_names]
        if _missing:
            _compat_errors.append(f'[{exp_name}] 누락 컬럼 {len(_missing)}개: {_missing[:10]}')
            continue

        _non_num = []
        for col in feat_list:
            _idx = _schema.get_field_index(col)
            if _idx >= 0:
                _t = str(_schema.field(_idx).type)
                if not any(k in _t for k in ['int', 'float', 'double', 'uint', 'decimal']):
                    _non_num.append(f'{col}:{_t}')
        if _non_num:
            _compat_errors.append(f'[{exp_name}] non-numeric feature: {_non_num[:5]}')
        else:
            print(f'[OK] {exp_name}.parquet: schema+dtype 확인')

        _sdf = pd.read_parquet(_pq_path, columns=['split', 'label'])
        _split_vals = set(_sdf['split'].dropna().astype(str).unique().tolist())
        if _split_vals != {'train', 'val', 'test'}:
            _compat_errors.append(f'[{exp_name}] split 값 이상: {sorted(_split_vals)}')
        if _sdf['split'].isna().any():
            _compat_errors.append(f'[{exp_name}] split null 존재')

        print(f'  [Label 분포] {exp_name}')
        for split_name in ['train', 'val', 'test']:
            _part = _sdf[_sdf['split'].astype(str) == split_name]
            _n = len(_part)
            _pos = int((_part['label'] == 1).sum()) if _n else 0
            print(f'    {split_name}: n={_n:,} pos={_pos:,} '
                  f'pos_ratio={(_pos/_n if _n else 0):.5f} both_classes={0 < _pos < _n}')
    except Exception as _e:
        _compat_errors.append(f'[{exp_name}] schema/label 검증 실패: {_e}')

# used_in_ml=True vs ml_exp00 단일 parquet 교차 확인
_exp00_full = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
if _exp00_full.exists():
    _exp00_cols = set(_pq.ParquetFile(_exp00_full).schema_arrow.names)
    _not_in = [c for c in _true_cols if c not in _exp00_cols]
    if _not_in:
        _compat_errors.append(f'[used_in_ml=True vs ml_exp00] 컬럼 없음: {_not_in}')
    else:
        print(f'[OK] used_in_ml=True {len(_true_cols)}개 컬럼 ml_exp00.parquet에 모두 존재')

# ml_io smoke test: 단일 parquet에서도 feature+label 로딩 가능 여부 확인
try:
    from ml_io import load_feature_columns as _lfc3, load_split as _ls3
    _smoke_feat = _lfc3(_feat_csv_path)
    if _exp00_full.exists():
        try:
            _Xl, _yl = _ls3(_exp00_full, _smoke_feat, label_col='label',
                            expected_split=None, sample_rows=50000, allow_nan=True)
            print(f'[OK] ml_io 단일 파일 smoke (50000행): X={_Xl.shape}, '
                  f'pos_ratio={_yl.mean():.5f} n_pos={int((_yl == 1).sum())}')
        except Exception as _e:
            _emsg = str(_e)
            if 'Both classes' in _emsg or 'both class' in _emsg.lower():
                print(f'[WARN] ml_io two-class smoke 실패: {_emsg}')
                print('       위 전체 label 분포를 참고하세요.')
            else:
                _compat_errors.append(f'[ml_io 단일 파일 smoke] 예상치 못한 실패: {_emsg}')
except ImportError:
    print('[SKIP] ml_io 없음 -> smoke test 건너뜀')

if _compat_errors:
    print('\n[COMPAT ERRORS]')
    for _e in _compat_errors:
        print(' -', _e)
    raise AssertionError(f'schema 오류 {len(_compat_errors)}건. 위 목록 확인.')
else:
    print('\n[COMPAT ALL OK] 단일 파일 parquet schema + dtype + split + used_in_ml=True 일치 확인')

used_in_ml=True 컬럼: 10개
[OK] ml_exp00.parquet: schema+dtype 확인
  [Label 분포] ml_exp00
    train: n=3,046,861 pos=2,297 pos_ratio=0.00075 both_classes=True
    val: n=1,015,920 pos=1,083 pos_ratio=0.00107 both_classes=True
    test: n=1,015,564 pos=1,797 pos_ratio=0.00177 both_classes=True
[OK] ml_exp00_no_time.parquet: schema+dtype 확인
  [Label 분포] ml_exp00_no_time
    train: n=3,046,861 pos=2,297 pos_ratio=0.00075 both_classes=True
    val: n=1,015,920 pos=1,083 pos_ratio=0.00107 both_classes=True
    test: n=1,015,564 pos=1,797 pos_ratio=0.00177 both_classes=True
[OK] used_in_ml=True 10개 컬럼 ml_exp00.parquet에 모두 존재
[SKIP] ml_io 없음 -> smoke test 건너뜀

[COMPAT ALL OK] 단일 파일 parquet schema + dtype + split + used_in_ml=True 일치 확인


## Optional ML 연동 smoke test: LR Baseline (ml_io + ml_metrics)

validation F1-optimal threshold -> test에 그대로 적용. Accuracy 단독 평가 금지.

In [17]:
# ============================================================
# Optional ML 연동 smoke test: LR Baseline (ml_io + ml_metrics)
#
# 목적: 최종 성능 판단이 아니라, 전처리 산출물이 ML파트 코드와 연결되는지 확인
# 기본값 False: 전처리 전달본을 가볍게 유지
# 필요 시 True로 바꾸고 실행하면 lr_baseline_result.json을 저장함
# ============================================================

RUN_OPTIONAL_LR_BASELINE = False

import sys, time, tracemalloc, json as _json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

_lr_json_path = CONFIG['DRIVE_OUTPUT_DIR'] / 'lr_baseline_result.json'
if not RUN_OPTIONAL_LR_BASELINE:
    if _lr_json_path.exists():
        _lr_json_path.unlink()  # optional 결과가 stale로 zip에 들어가지 않도록 제거
    print('[SKIP] Optional LR Baseline은 기본 실행하지 않습니다.')
    print('       필요 시 RUN_OPTIONAL_LR_BASELINE=True로 바꾸고 이 셀을 실행하세요.')
else:
    # ── ml_io / ml_metrics import smoke test ──────────────────────────
    try:
        from ml_io import load_feature_columns as _lfc, load_split as _ls
        from ml_metrics import select_threshold_by_f1, evaluate_at_threshold
        _HAS_MOD = True
        print('[OK] ml_io, ml_metrics import smoke test 통과')
    except ImportError as _e:
        _HAS_MOD = False
        print(f'[WARN] ml_io/ml_metrics import 실패: {_e} -> sklearn fallback')

    # ── feature 컬럼 로드 ─────────────────────────────────────────────
    _feat_csv = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_feature_columns.csv'
    if not _feat_csv.exists():
        raise FileNotFoundError(f'ml_feature_columns.csv 없음: {_feat_csv}')

    if _HAS_MOD:
        LR_FEAT = _lfc(_feat_csv)
        print(f'ml_feature_columns.csv -> {len(LR_FEAT)}개 feature')
    else:
        _fc_df = pd.read_csv(_feat_csv, encoding='utf-8-sig')
        LR_FEAT = _fc_df.loc[_fc_df['used_in_ml'] == True, 'column_name'].tolist()
        print(f'[FALLBACK pandas] ml_feature_columns.csv -> {len(LR_FEAT)}개 feature')

    _missing_in_exp00 = [c for c in LR_FEAT if c not in ml_exp00.columns]
    if _missing_in_exp00:
        raise ValueError('[LR ERROR] used_in_ml=True 컬럼이 ml_exp00에 없습니다:\n'
                         + '\n'.join(f'  - {c}' for c in _missing_in_exp00))

    # 단일 parquet smoke: train/val/test 파일을 요구하지 않음
    if _HAS_MOD:
        _full_pq = CONFIG['DRIVE_OUTPUT_DIR'] / 'ml_exp00.parquet'
        if _full_pq.exists():
            try:
                _X_s, _y_s = _ls(_full_pq, LR_FEAT, label_col='label',
                                  expected_split=None, sample_rows=50000,
                                  allow_nan=True)
                print(f'[OK] ml_io 단일 파일 sanity (50000행): X={_X_s.shape}, '
                      f'pos_ratio={_y_s.mean():.5f}')
            except Exception as _e:
                _emsg = str(_e)
                if ('Both classes' in _emsg or
                        'both class' in _emsg.lower() or
                        'one class' in _emsg.lower()):
                    print(f'[WARN] ml_io sanity one-class sample: {_emsg}')
                else:
                    raise

    def _Xy(df_pl, feats, split_name):
        sub = df_pl.filter(pl.col('split') == split_name)
        return (sub.select(feats).to_pandas().fillna(0),
                sub.select('label').to_pandas().squeeze().astype(int))

    X_tr, y_tr = _Xy(ml_exp00, LR_FEAT, 'train')
    X_va, y_va = _Xy(ml_exp00, LR_FEAT, 'val')
    X_te, y_te = _Xy(ml_exp00, LR_FEAT, 'test')
    print(f'train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}')

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s, X_te_s = sc.transform(X_va), sc.transform(X_te)

    tracemalloc.start()
    t0 = time.time()
    lr = LogisticRegression(max_iter=1000, class_weight='balanced',
                            solver='lbfgs', random_state=CONFIG['RANDOM_SEED'])
    lr.fit(X_tr_s, y_tr)
    fit_t = time.time() - t0
    _cur_mem, _peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    va_p = lr.predict_proba(X_va_s)[:, 1]
    te_p = lr.predict_proba(X_te_s)[:, 1]

    if _HAS_MOD:
        best_thr = select_threshold_by_f1(y_va, va_p)['threshold']
        va_m = evaluate_at_threshold(y_va, va_p, best_thr)['summary']
        te_m = evaluate_at_threshold(y_te, te_p, best_thr)['summary']
    else:
        from sklearn.metrics import (precision_recall_curve, f1_score,
            recall_score, precision_score, average_precision_score, confusion_matrix)
        _pr, _re, _th = precision_recall_curve(y_va, va_p)
        _f1s = 2 * _pr[:-1] * _re[:-1] / (_pr[:-1] + _re[:-1] + 1e-9)
        best_thr = float(_th[_f1s.argmax()])
        def _ev(y, p, t):
            _pd = (p >= t).astype(int)
            _tn, _fp, _fn, _tp = confusion_matrix(y, _pd, labels=[0, 1]).ravel()
            return {'threshold': t, 'f1': f1_score(y, _pd, zero_division=0),
                    'recall': recall_score(y, _pd, zero_division=0),
                    'precision': precision_score(y, _pd, zero_division=0),
                    'average_precision': average_precision_score(y, p),
                    'tn': int(_tn), 'fp': int(_fp), 'fn': int(_fn), 'tp': int(_tp)}
        va_m, te_m = _ev(y_va, va_p, best_thr), _ev(y_te, te_p, best_thr)

    t2 = time.time()
    lr.predict_proba(X_te_s)
    inf_t = time.time() - t2

    print('\n===== Optional LR Baseline =====')
    print(f'n_features={len(LR_FEAT)}, threshold={best_thr:.4f} (val F1-opt)')
    print(f'[VAL]  F1={va_m["f1"]:.4f} Recall={va_m["recall"]:.4f} '
          f'Precision={va_m["precision"]:.4f} AP={va_m["average_precision"]:.4f}')
    print(f'[TEST] F1={te_m["f1"]:.4f} Recall={te_m["recall"]:.4f} '
          f'Precision={te_m["precision"]:.4f} AP={te_m["average_precision"]:.4f}')

    _res = {
        'model': 'LogisticRegression',
        'feature_set': 'ml_exp00',
        'n_features': len(LR_FEAT),
        'threshold': best_thr,
        'fit_time_sec': round(fit_t, 4),
        'inference_time_test_sec': round(inf_t, 6),
        'memory_usage_mb': round(_cur_mem / 1024 / 1024, 2),
        'peak_memory_mb': round(_peak_mem / 1024 / 1024, 2),
        'val': va_m,
        'test': te_m,
        'note': 'optional ML 연동 smoke test; not final performance judgment',
    }
    with open(_lr_json_path, 'w', encoding='utf-8') as _f:
        _json.dump(_res, _f, indent=2, ensure_ascii=False)
    print('saved:', _lr_json_path)


[SKIP] Optional LR Baseline은 기본 실행하지 않습니다.
       필요 시 RUN_OPTIONAL_LR_BASELINE=True로 바꾸고 이 셀을 실행하세요.


## 03_ml_feature_process 요약  
### 핵심 원칙

| 항목 | 내용 |
|------|------|
| parquet 컬럼명 | **기존 전처리파트 output 유지** (`amount__current__value`, `cat__from_bank__code` 등) |
| split | `split` 컬럼으로 train/val/test를 구분합니다. split은 한 번 확정 후 `split_assignment.csv`로 고정·재사용합니다. |
| 전달 파일 형태 | train/val/test 분리 파일이 아니라, **하나의 parquet 안에 split 컬럼을 포함**합니다. |
| categorical encoding | 5개 컬럼 train-only code encoding. val/test unseen → -1 |

### 최종 전달 parquet

| 파일 | 설명 |
|------|------|
| `ml_exp00.parquet` | 기본 ML-00 feature. train/val/test가 `split` 컬럼으로 함께 들어 있음 |
| `ml_exp00_no_time.parquet` | time feature 제외 비교용. train/val/test가 `split` 컬럼으로 함께 들어 있음 |

### feature 사용 기준 (`ml_feature_columns.csv`)

`column_name`은 실제 parquet 컬럼명 기준입니다.

| feature 그룹 | `used_in_ml` 기본값 | 비고 |
|---|---|---|
| ml_exp00 기본 feature (금액 2 + cat 5 + 시간 3) | **True** | 기본 전달 기준 |
| pair feature (`cat__currency_pair__code` 등) | False | parquet에 포함. 성능 비교 후 True로 변경 가능 |
| `amount_received` 계열 | False | clean_base에 amount_received 있을 때만 생성됨 |
| ML-01 history feature | False | history feature 실험 활성화 시 True로 변경 |

### Optional LR Baseline

- LR Baseline은 최종 성능 판단이 아니라 **ML파트 연동 확인용 smoke test**입니다.
- 기본 실행하지 않으며, 필요할 때 `RUN_OPTIONAL_LR_BASELINE=True`로 바꿔 실행할 수 있습니다.
- 최종 성능 판단, 모델 튜닝, threshold 정책 결정은 ML파트에서 진행합니다.

### ML파트가 조정할 수 있는 항목

- `ml_feature_columns.csv`에서 `used_in_ml` 컬럼을 True/False로 변경하면 모델 입력이 바뀝니다.
- ML파트 코드는 단일 parquet를 읽은 뒤 `split` 컬럼으로 train/val/test를 분리하면 됩니다.
- **time feature 최종 사용 여부**: `ml_exp00` vs `ml_exp00_no_time` 비교로 결정 가능.
- **pair feature 활성화**: `cat__currency_pair__code`, `cat__bank_pair__code` → `used_in_ml=True`로 변경.
- **amount_received 계열**: clean_base에 컬럼 있고 성능 향상 확인 시 활성화.
- ML-01 history feature 실험: `ml_feature_columns.csv`에서 history feature `used_in_ml=True`로 변경.

### 산출물 위치

| 파일 | 위치 | 설명 |
|------|------|------|
| `ml_exp00.parquet` 등 | Drive/ml_features | 단일 parquet 학습 데이터 (`split` 컬럼 포함) |
| `ml_feature_columns.csv` | Drive/ml_features | **feature 활성화 기준** |
| `feature_catalog.csv` | Drive/ml_features | 전체 feature 정의 |

> **GNN/PageRank/경량화/모델 튜닝/feature 최종 선택은 이번 범위에서 제외합니다.**
